<a href="https://colab.research.google.com/github/SBU2/food-race/blob/main/Mine_Rehabilitation_Financial_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mine Rehabilitation Financial Investment Analysis

## Objective 3: To analyse the financial investments made in rehabilitation projects of the selected mines

This notebook implements comprehensive financial analysis methods for mine rehabilitation projects, including:
1. Economic Data Analysis
2. Correlation and Regression Analysis
3. Cost-Benefit Analysis
4. Econometric Methods


## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
%matplotlib inline

## 2. Generate Sample Mine Rehabilitation Data

Since real data may not be available, we'll generate representative sample data for demonstration purposes.

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Generate sample data for 5 mines over 10 years (2014-2023)
n_mines = 5
n_years = 10
mine_names = [f'Mine_{chr(65+i)}' for i in range(n_mines)]
years = list(range(2014, 2024))

# Create comprehensive dataset
data = []
for mine in mine_names:
    for year in years:
        # Base values with some mine-specific characteristics
        mine_factor = np.random.normal(1, 0.2)
        year_trend = 1 + (year - 2014) * 0.05  # 5% annual growth trend
        
        # Financial data (in millions USD)
        mining_revenue = np.random.normal(50, 10) * mine_factor * year_trend
        mining_profit = mining_revenue * np.random.uniform(0.15, 0.25)
        
        # Rehabilitation investments
        rehab_investment = np.random.normal(5, 1.5) * mine_factor * year_trend
        
        # Area and production data
        disturbed_area = np.random.normal(100, 20)  # hectares
        ore_production = np.random.normal(500, 100) * mine_factor  # thousand tons
        
        # Effectiveness ratings (1-10 scale)
        # Higher investment should correlate with better effectiveness
        base_effectiveness = 3 + (rehab_investment / 10) * 5
        effectiveness_rating = np.clip(np.random.normal(base_effectiveness, 1), 1, 10)
        
        # Ecological recovery indicators (0-1 scale)
        vegetation_recovery = np.clip(np.random.beta(2, 3) + (rehab_investment / 20), 0, 1)
        soil_quality_index = np.clip(np.random.beta(2, 3) + (rehab_investment / 25), 0, 1)
        water_quality_index = np.clip(np.random.beta(2, 3) + (rehab_investment / 30), 0, 1)
        
        # Ecosystem service values (estimated)
        ecosystem_value = (
            vegetation_recovery * 10000 + 
            soil_quality_index * 8000 + 
            water_quality_index * 12000
        ) * disturbed_area / 100
        
        data.append({
            'Mine': mine,
            'Year': year,
            'Mining_Revenue_M': mining_revenue,
            'Mining_Profit_M': mining_profit,
            'Rehabilitation_Investment_M': rehab_investment,
            'Disturbed_Area_Ha': disturbed_area,
            'Ore_Production_KTons': ore_production,
            'Effectiveness_Rating': effectiveness_rating,
            'Vegetation_Recovery': vegetation_recovery,
            'Soil_Quality_Index': soil_quality_index,
            'Water_Quality_Index': water_quality_index,
            'Ecosystem_Service_Value': ecosystem_value
        })

# Create DataFrame
df = pd.DataFrame(data)

# Calculate derived metrics
df['Investment_Per_Area'] = df['Rehabilitation_Investment_M'] / df['Disturbed_Area_Ha'] * 1000  # per hectare
df['Investment_Per_Ton'] = df['Rehabilitation_Investment_M'] / df['Ore_Production_KTons']  # per thousand tons
df['Investment_Revenue_Ratio'] = df['Rehabilitation_Investment_M'] / df['Mining_Revenue_M'] * 100  # percentage
df['Investment_Profit_Ratio'] = df['Rehabilitation_Investment_M'] / df['Mining_Profit_M'] * 100  # percentage

print("Sample Mine Rehabilitation Dataset Generated")
print(f"Shape: {df.shape}")
print("\nFirst few rows:")
df.head()

## 3. Economic Data Analysis

### 3.1 Descriptive Statistics

In [ ]:
# Calculate comprehensive descriptive statistics
print("=== DESCRIPTIVE STATISTICS FOR REHABILITATION INVESTMENTS ===")
print("\n1. Total Rehabilitation Investments:")
total_investment = df['Rehabilitation_Investment_M'].sum()
print(f"Total across all mines and years: ${total_investment:.2f} million")

print("\n2. Per-Area Investment Statistics:")
per_area_stats = df['Investment_Per_Area'].describe()
print(per_area_stats)
print(f"\nMean investment per hectare: ${per_area_stats['mean']:.2f}K")
print(f"Median investment per hectare: ${per_area_stats['50%']:.2f}K")

print("\n3. Per-Ton Investment Statistics:")
per_ton_stats = df['Investment_Per_Ton'].describe()
print(per_ton_stats)
print(f"\nMean investment per thousand tons: ${per_ton_stats['mean']:.4f}M")
print(f"Median investment per thousand tons: ${per_ton_stats['50%']:.4f}M")

print("\n4. Investment by Mine:")
investment_by_mine = df.groupby('Mine')['Rehabilitation_Investment_M'].agg(['sum', 'mean', 'std']).round(2)
print(investment_by_mine)

print("\n5. Investment Ratios:")
ratio_stats = df[['Investment_Revenue_Ratio', 'Investment_Profit_Ratio']].describe().round(2)
print(ratio_stats)

### 3.2 Trend Analysis - Time Series Analysis

In [ ]:
# Time series analysis of rehabilitation investments
print("=== TREND ANALYSIS - TIME SERIES ANALYSIS ===")

# Aggregate data by year
yearly_data = df.groupby('Year').agg({
    'Rehabilitation_Investment_M': ['sum', 'mean'],
    'Investment_Per_Area': 'mean',
    'Investment_Revenue_Ratio': 'mean'
}).round(2)

yearly_data.columns = ['Total_Investment', 'Avg_Investment', 'Avg_Per_Area', 'Avg_Revenue_Ratio']
print("\nYearly Investment Trends:")
print(yearly_data)

# Calculate year-over-year growth rates
yearly_data['Investment_Growth_Rate'] = yearly_data['Total_Investment'].pct_change() * 100
print("\nYear-over-Year Growth Rates:")
print(yearly_data[['Total_Investment', 'Investment_Growth_Rate']].round(2))

# Linear trend analysis
years_numeric = np.array(yearly_data.index)
investment_values = yearly_data['Total_Investment'].values

# Fit linear trend
slope, intercept, r_value, p_value, std_err = stats.linregress(years_numeric, investment_values)

print(f"\nLinear Trend Analysis:")
print(f"Slope: ${slope:.2f}M per year")
print(f"R-squared: {r_value**2:.3f}")
print(f"P-value: {p_value:.3f}")
print(f"Annual growth trend: {'Increasing' if slope > 0 else 'Decreasing'}")

# Create trend visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Plot 1: Total investment over time
axes[0,0].plot(yearly_data.index, yearly_data['Total_Investment'], 'o-', linewidth=2, markersize=8)
axes[0,0].plot(yearly_data.index, slope * years_numeric + intercept, '--', alpha=0.7, label=f'Trend (slope={slope:.2f})')
axes[0,0].set_title('Total Rehabilitation Investment Over Time')
axes[0,0].set_xlabel('Year')
axes[0,0].set_ylabel('Investment ($ Million)')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Plot 2: Average investment per area
axes[0,1].plot(yearly_data.index, yearly_data['Avg_Per_Area'], 's-', linewidth=2, markersize=8, color='orange')
axes[0,1].set_title('Average Investment per Hectare Over Time')
axes[0,1].set_xlabel('Year')
axes[0,1].set_ylabel('Investment ($K per hectare)')
axes[0,1].grid(True, alpha=0.3)

# Plot 3: Investment as % of revenue
axes[1,0].plot(yearly_data.index, yearly_data['Avg_Revenue_Ratio'], '^-', linewidth=2, markersize=8, color='green')
axes[1,0].set_title('Investment as % of Revenue Over Time')
axes[1,0].set_xlabel('Year')
axes[1,0].set_ylabel('Investment/Revenue Ratio (%)')
axes[1,0].grid(True, alpha=0.3)

# Plot 4: Growth rates
growth_data = yearly_data['Investment_Growth_Rate'].dropna()
axes[1,1].bar(growth_data.index, growth_data.values, alpha=0.7, color='red')
axes[1,1].axhline(y=0, color='black', linestyle='-', alpha=0.5)
axes[1,1].set_title('Year-over-Year Investment Growth Rate')
axes[1,1].set_xlabel('Year')
axes[1,1].set_ylabel('Growth Rate (%)')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Stationarity test
adf_result = adfuller(investment_values)
print(f"\nStationarity Test (Augmented Dickey-Fuller):")
print(f"ADF Statistic: {adf_result[0]:.3f}")
print(f"P-value: {adf_result[1]:.3f}")
print(f"Series is {'stationary' if adf_result[1] < 0.05 else 'non-stationary'} at 5% significance level")

### 3.3 Ratio Analysis

In [ ]:
# Comprehensive ratio analysis
print("=== RATIO ANALYSIS ===")

# 1. Investment as percentage of mining revenue
print("\n1. Rehabilitation Investment as % of Mining Revenue:")
revenue_ratio_stats = df['Investment_Revenue_Ratio'].describe()
print(revenue_ratio_stats.round(2))

# 2. Investment as percentage of mining profit
print("\n2. Rehabilitation Investment as % of Mining Profit:")
profit_ratio_stats = df['Investment_Profit_Ratio'].describe()
print(profit_ratio_stats.round(2))

# 3. Mine-specific ratio analysis
print("\n3. Ratio Analysis by Mine:")
mine_ratios = df.groupby('Mine')[['Investment_Revenue_Ratio', 'Investment_Profit_Ratio']].agg(['mean', 'std']).round(2)
print(mine_ratios)

# 4. Time trend in ratios
print("\n4. Ratio Trends Over Time:")
ratio_trends = df.groupby('Year')[['Investment_Revenue_Ratio', 'Investment_Profit_Ratio']].mean().round(2)
print(ratio_trends)

# Visualization of ratios
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Plot 1: Investment/Revenue ratio distribution
axes[0,0].hist(df['Investment_Revenue_Ratio'], bins=20, alpha=0.7, edgecolor='black')
axes[0,0].axvline(df['Investment_Revenue_Ratio'].mean(), color='red', linestyle='--', 
                  label=f'Mean: {df["Investment_Revenue_Ratio"].mean():.1f}%')
axes[0,0].set_title('Distribution of Investment/Revenue Ratio')
axes[0,0].set_xlabel('Investment as % of Revenue')
axes[0,0].set_ylabel('Frequency')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Plot 2: Investment/Profit ratio distribution
axes[0,1].hist(df['Investment_Profit_Ratio'], bins=20, alpha=0.7, color='orange', edgecolor='black')
axes[0,1].axvline(df['Investment_Profit_Ratio'].mean(), color='red', linestyle='--',
                  label=f'Mean: {df["Investment_Profit_Ratio"].mean():.1f}%')
axes[0,1].set_title('Distribution of Investment/Profit Ratio')
axes[0,1].set_xlabel('Investment as % of Profit')
axes[0,1].set_ylabel('Frequency')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# Plot 3: Ratio trends over time
axes[1,0].plot(ratio_trends.index, ratio_trends['Investment_Revenue_Ratio'], 'o-', 
               label='Revenue Ratio', linewidth=2, markersize=8)
axes[1,0].plot(ratio_trends.index, ratio_trends['Investment_Profit_Ratio'], 's-', 
               label='Profit Ratio', linewidth=2, markersize=8)
axes[1,0].set_title('Investment Ratios Over Time')
axes[1,0].set_xlabel('Year')
axes[1,0].set_ylabel('Ratio (%)')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# Plot 4: Box plot of ratios by mine
mine_names = df['Mine'].unique()
ratio_data = [df[df['Mine'] == mine]['Investment_Revenue_Ratio'].values for mine in mine_names]
axes[1,1].boxplot(ratio_data, labels=mine_names)
axes[1,1].set_title('Investment/Revenue Ratio by Mine')
axes[1,1].set_xlabel('Mine')
axes[1,1].set_ylabel('Investment/Revenue Ratio (%)')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistical tests for ratio differences
print("\n5. Statistical Tests:")
# ANOVA test for differences in ratios between mines
mine_groups = [df[df['Mine'] == mine]['Investment_Revenue_Ratio'].values for mine in mine_names]
f_stat, p_value = stats.f_oneway(*mine_groups)
print(f"ANOVA test for revenue ratio differences between mines:")
print(f"F-statistic: {f_stat:.3f}, P-value: {p_value:.3f}")
print(f"Significant differences between mines: {'Yes' if p_value < 0.05 else 'No'}")

## 4. Correlation and Regression Analysis

### 4.1 Correlation Analysis

In [ ]:
# Correlation analysis between financial investments and rehabilitation effectiveness
print("=== CORRELATION ANALYSIS ===")

# Select relevant variables for correlation analysis
correlation_vars = [
    'Rehabilitation_Investment_M', 'Investment_Per_Area', 'Investment_Per_Ton',
    'Effectiveness_Rating', 'Vegetation_Recovery', 'Soil_Quality_Index', 
    'Water_Quality_Index', 'Ecosystem_Service_Value'
]

# Calculate correlation matrix
correlation_matrix = df[correlation_vars].corr()

print("\n1. Correlation Matrix:")
print(correlation_matrix.round(3))

# Focus on investment-effectiveness correlations
print("\n2. Key Investment-Effectiveness Correlations:")
investment_vars = ['Rehabilitation_Investment_M', 'Investment_Per_Area', 'Investment_Per_Ton']
effectiveness_vars = ['Effectiveness_Rating', 'Vegetation_Recovery', 'Soil_Quality_Index', 'Water_Quality_Index']

for inv_var in investment_vars:
    print(f"\n{inv_var}:")
    for eff_var in effectiveness_vars:
        corr_coef = correlation_matrix.loc[inv_var, eff_var]
        # Calculate p-value
        _, p_value = stats.pearsonr(df[inv_var], df[eff_var])
        significance = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else ""
        print(f"  vs {eff_var}: {corr_coef:.3f}{significance} (p={p_value:.3f})")

# Visualization of correlation matrix
plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, cmap='RdYlBu_r', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": .8})
plt.title('Correlation Matrix: Investment and Effectiveness Variables')
plt.tight_layout()
plt.show()

# Scatter plots for key relationships
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Plot 1: Investment vs Effectiveness Rating
axes[0,0].scatter(df['Rehabilitation_Investment_M'], df['Effectiveness_Rating'], alpha=0.6)
z = np.polyfit(df['Rehabilitation_Investment_M'], df['Effectiveness_Rating'], 1)
p = np.poly1d(z)
axes[0,0].plot(df['Rehabilitation_Investment_M'], p(df['Rehabilitation_Investment_M']), "r--", alpha=0.8)
axes[0,0].set_xlabel('Rehabilitation Investment ($M)')
axes[0,0].set_ylabel('Effectiveness Rating')
axes[0,0].set_title('Investment vs Effectiveness Rating')
axes[0,0].grid(True, alpha=0.3)

# Plot 2: Investment vs Vegetation Recovery
axes[0,1].scatter(df['Rehabilitation_Investment_M'], df['Vegetation_Recovery'], alpha=0.6, color='green')
z = np.polyfit(df['Rehabilitation_Investment_M'], df['Vegetation_Recovery'], 1)
p = np.poly1d(z)
axes[0,1].plot(df['Rehabilitation_Investment_M'], p(df['Rehabilitation_Investment_M']), "r--", alpha=0.8)
axes[0,1].set_xlabel('Rehabilitation Investment ($M)')
axes[0,1].set_ylabel('Vegetation Recovery Index')
axes[0,1].set_title('Investment vs Vegetation Recovery')
axes[0,1].grid(True, alpha=0.3)

# Plot 3: Per-area investment vs Soil Quality
axes[1,0].scatter(df['Investment_Per_Area'], df['Soil_Quality_Index'], alpha=0.6, color='brown')
z = np.polyfit(df['Investment_Per_Area'], df['Soil_Quality_Index'], 1)
p = np.poly1d(z)
axes[1,0].plot(df['Investment_Per_Area'], p(df['Investment_Per_Area']), "r--", alpha=0.8)
axes[1,0].set_xlabel('Investment per Area ($K/ha)')
axes[1,0].set_ylabel('Soil Quality Index')
axes[1,0].set_title('Per-Area Investment vs Soil Quality')
axes[1,0].grid(True, alpha=0.3)

# Plot 4: Investment vs Ecosystem Service Value
axes[1,1].scatter(df['Rehabilitation_Investment_M'], df['Ecosystem_Service_Value'], alpha=0.6, color='purple')
z = np.polyfit(df['Rehabilitation_Investment_M'], df['Ecosystem_Service_Value'], 1)
p = np.poly1d(z)
axes[1,1].plot(df['Rehabilitation_Investment_M'], p(df['Rehabilitation_Investment_M']), "r--", alpha=0.8)
axes[1,1].set_xlabel('Rehabilitation Investment ($M)')
axes[1,1].set_ylabel('Ecosystem Service Value')
axes[1,1].set_title('Investment vs Ecosystem Service Value')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 4.2 Multiple Regression Analysis

In [ ]:
# Multiple regression analysis
print("=== MULTIPLE REGRESSION ANALYSIS ===")

# Model 1: Predict Effectiveness Rating from Investment Variables
print("\n1. Model 1: Effectiveness Rating = f(Investment Variables)")

# Prepare data
X1 = df[['Rehabilitation_Investment_M', 'Investment_Per_Area', 'Mining_Revenue_M']]
y1 = df['Effectiveness_Rating']

# Add constant term
X1_const = sm.add_constant(X1)

# Fit model
model1 = sm.OLS(y1, X1_const).fit()
print(model1.summary())

# Model diagnostics
print("\nModel 1 Diagnostics:")
print(f"R-squared: {model1.rsquared:.3f}")
print(f"Adjusted R-squared: {model1.rsquared_adj:.3f}")
print(f"F-statistic: {model1.fvalue:.3f}")
print(f"F-statistic p-value: {model1.f_pvalue:.3f}")

# Heteroscedasticity test
bp_test = het_breuschpagan(model1.resid, model1.model.exog)
print(f"Breusch-Pagan test p-value: {bp_test[1]:.3f} ({'Heteroscedastic' if bp_test[1] < 0.05 else 'Homoscedastic'})")

# Model 2: Predict Ecological Recovery Composite Index
print("\n\n2. Model 2: Ecological Recovery Index = f(Investment and Control Variables)")

# Create ecological recovery composite index
df['Ecological_Recovery_Index'] = (
    df['Vegetation_Recovery'] * 0.4 + 
    df['Soil_Quality_Index'] * 0.3 + 
    df['Water_Quality_Index'] * 0.3
)

# Prepare data
X2 = df[['Rehabilitation_Investment_M', 'Investment_Per_Area', 'Disturbed_Area_Ha', 'Year']]
y2 = df['Ecological_Recovery_Index']

# Add constant term
X2_const = sm.add_constant(X2)

# Fit model
model2 = sm.OLS(y2, X2_const).fit()
print(model2.summary())

# Model 3: Predict Ecosystem Service Value
print("\n\n3. Model 3: Ecosystem Service Value = f(Investment and Ecological Indicators)")

X3 = df[['Rehabilitation_Investment_M', 'Vegetation_Recovery', 'Soil_Quality_Index', 'Water_Quality_Index']]
y3 = df['Ecosystem_Service_Value']

X3_const = sm.add_constant(X3)
model3 = sm.OLS(y3, X3_const).fit()
print(model3.summary())

# Visualization of regression results
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Model 1 plots
axes[0,0].scatter(model1.fittedvalues, model1.resid, alpha=0.6)
axes[0,0].axhline(y=0, color='red', linestyle='--')
axes[0,0].set_xlabel('Fitted Values')
axes[0,0].set_ylabel('Residuals')
axes[0,0].set_title('Model 1: Residuals vs Fitted')
axes[0,0].grid(True, alpha=0.3)

axes[0,1].scatter(y1, model1.fittedvalues, alpha=0.6)
axes[0,1].plot([y1.min(), y1.max()], [y1.min(), y1.max()], 'r--', lw=2)
axes[0,1].set_xlabel('Actual Effectiveness Rating')
axes[0,1].set_ylabel('Predicted Effectiveness Rating')
axes[0,1].set_title('Model 1: Actual vs Predicted')
axes[0,1].grid(True, alpha=0.3)

# Q-Q plot for Model 1
stats.probplot(model1.resid, dist="norm", plot=axes[0,2])
axes[0,2].set_title('Model 1: Q-Q Plot')
axes[0,2].grid(True, alpha=0.3)

# Model 2 plots
axes[1,0].scatter(model2.fittedvalues, model2.resid, alpha=0.6, color='green')
axes[1,0].axhline(y=0, color='red', linestyle='--')
axes[1,0].set_xlabel('Fitted Values')
axes[1,0].set_ylabel('Residuals')
axes[1,0].set_title('Model 2: Residuals vs Fitted')
axes[1,0].grid(True, alpha=0.3)

axes[1,1].scatter(y2, model2.fittedvalues, alpha=0.6, color='green')
axes[1,1].plot([y2.min(), y2.max()], [y2.min(), y2.max()], 'r--', lw=2)
axes[1,1].set_xlabel('Actual Ecological Recovery Index')
axes[1,1].set_ylabel('Predicted Ecological Recovery Index')
axes[1,1].set_title('Model 2: Actual vs Predicted')
axes[1,1].grid(True, alpha=0.3)

# Feature importance (coefficients)
coef_data = pd.DataFrame({
    'Variable': X1.columns,
    'Coefficient': model1.params[1:],  # Exclude constant
    'P_value': model1.pvalues[1:]
})
coef_data = coef_data.sort_values('Coefficient', key=abs, ascending=False)

bars = axes[1,2].bar(range(len(coef_data)), coef_data['Coefficient'], 
                     color=['red' if p < 0.05 else 'blue' for p in coef_data['P_value']],
                     alpha=0.7)
axes[1,2].set_xticks(range(len(coef_data)))
axes[1,2].set_xticklabels(coef_data['Variable'], rotation=45, ha='right')
axes[1,2].set_ylabel('Coefficient Value')
axes[1,2].set_title('Model 1: Coefficient Importance\n(Red = Significant, Blue = Non-significant)')
axes[1,2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary of all models
print("\n=== REGRESSION MODELS SUMMARY ===")
models_summary = pd.DataFrame({
    'Model': ['Effectiveness Rating', 'Ecological Recovery Index', 'Ecosystem Service Value'],
    'R_squared': [model1.rsquared, model2.rsquared, model3.rsquared],
    'Adj_R_squared': [model1.rsquared_adj, model2.rsquared_adj, model3.rsquared_adj],
    'F_statistic': [model1.fvalue, model2.fvalue, model3.fvalue],
    'F_p_value': [model1.f_pvalue, model2.f_pvalue, model3.f_pvalue]
})
print(models_summary.round(4))

## 5. Cost-Benefit Analysis

### 5.1 Rehabilitation Costs vs Ecosystem Service Values

In [ ]:
# Cost-Benefit Analysis
print("=== COST-BENEFIT ANALYSIS ===")

# Calculate basic cost-benefit ratios
df['Benefit_Cost_Ratio'] = df['Ecosystem_Service_Value'] / (df['Rehabilitation_Investment_M'] * 1000)  # Investment in thousands

print("\n1. Basic Cost-Benefit Analysis:")
print(f"Mean Benefit-Cost Ratio: {df['Benefit_Cost_Ratio'].mean():.2f}")
print(f"Median Benefit-Cost Ratio: {df['Benefit_Cost_Ratio'].median():.2f}")
print(f"Percentage of projects with B/C > 1: {(df['Benefit_Cost_Ratio'] > 1).mean() * 100:.1f}%")

# Cost-benefit statistics by mine
cb_by_mine = df.groupby('Mine')['Benefit_Cost_Ratio'].agg(['mean', 'median', 'min', 'max']).round(2)
print("\n2. Cost-Benefit Ratios by Mine:")
print(cb_by_mine)

# Time trend in cost-benefit ratios
cb_by_year = df.groupby('Year')['Benefit_Cost_Ratio'].mean()
print("\n3. Cost-Benefit Ratio Trends Over Time:")
print(cb_by_year.round(2))

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Plot 1: Distribution of B/C ratios
axes[0,0].hist(df['Benefit_Cost_Ratio'], bins=20, alpha=0.7, edgecolor='black')
axes[0,0].axvline(1, color='red', linestyle='--', label='Break-even (B/C = 1)')
axes[0,0].axvline(df['Benefit_Cost_Ratio'].mean(), color='orange', linestyle='-', 
                  label=f'Mean: {df["Benefit_Cost_Ratio"].mean():.2f}')
axes[0,0].set_xlabel('Benefit-Cost Ratio')
axes[0,0].set_ylabel('Frequency')
axes[0,0].set_title('Distribution of Benefit-Cost Ratios')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Plot 2: B/C ratio over time
axes[0,1].plot(cb_by_year.index, cb_by_year.values, 'o-', linewidth=2, markersize=8)
axes[0,1].axhline(1, color='red', linestyle='--', alpha=0.7, label='Break-even')
axes[0,1].set_xlabel('Year')
axes[0,1].set_ylabel('Benefit-Cost Ratio')
axes[0,1].set_title('Cost-Benefit Ratio Trends Over Time')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# Plot 3: Investment vs Benefits scatter
axes[1,0].scatter(df['Rehabilitation_Investment_M'], df['Ecosystem_Service_Value'], 
                  c=df['Benefit_Cost_Ratio'], cmap='RdYlGn', alpha=0.7)
cbar = plt.colorbar(axes[1,0].collections[0], ax=axes[1,0])
cbar.set_label('B/C Ratio')
axes[1,0].set_xlabel('Rehabilitation Investment ($M)')
axes[1,0].set_ylabel('Ecosystem Service Value')
axes[1,0].set_title('Investment vs Benefits (colored by B/C ratio)')
axes[1,0].grid(True, alpha=0.3)

# Plot 4: B/C ratio by mine
mine_names = df['Mine'].unique()
bc_data = [df[df['Mine'] == mine]['Benefit_Cost_Ratio'].values for mine in mine_names]
axes[1,1].boxplot(bc_data, labels=mine_names)
axes[1,1].axhline(1, color='red', linestyle='--', alpha=0.7, label='Break-even')
axes[1,1].set_xlabel('Mine')
axes[1,1].set_ylabel('Benefit-Cost Ratio')
axes[1,1].set_title('Benefit-Cost Ratio by Mine')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistical analysis of cost-effectiveness
print("\n4. Statistical Analysis:")
# Test if B/C ratios are significantly different from 1
t_stat, p_value = stats.ttest_1samp(df['Benefit_Cost_Ratio'], 1)
print(f"One-sample t-test (H0: B/C = 1):")
print(f"T-statistic: {t_stat:.3f}, P-value: {p_value:.3f}")
print(f"Conclusion: B/C ratios are {'significantly different from' if p_value < 0.05 else 'not significantly different from'} 1")

# Correlation between investment amount and cost-effectiveness
corr_inv_bc, p_corr = stats.pearsonr(df['Rehabilitation_Investment_M'], df['Benefit_Cost_Ratio'])
print(f"\nCorrelation between Investment Amount and B/C Ratio: {corr_inv_bc:.3f} (p={p_corr:.3f})")

### 5.2 Net Present Value (NPV) Analysis

In [ ]:
# Net Present Value Analysis for Long-term Rehabilitation Investments
print("=== NET PRESENT VALUE (NPV) ANALYSIS ===")

# Parameters for NPV calculation
discount_rate = 0.05  # 5% discount rate
project_lifespan = 20  # 20 years
annual_degradation = 0.02  # 2% annual degradation of ecosystem services

def calculate_npv(initial_investment, annual_benefit, discount_rate, years, degradation_rate=0):
    """Calculate NPV with optional benefit degradation over time"""
    npv = -initial_investment  # Initial cost
    for year in range(1, years + 1):
        # Benefits degrade over time if degradation_rate > 0
        annual_benefit_adjusted = annual_benefit * ((1 - degradation_rate) ** (year - 1))
        discounted_benefit = annual_benefit_adjusted / ((1 + discount_rate) ** year)
        npv += discounted_benefit
    return npv

# Calculate NPV for each rehabilitation project
print(f"\nNPV Analysis Parameters:")
print(f"Discount Rate: {discount_rate*100}%")
print(f"Project Lifespan: {project_lifespan} years")
print(f"Annual Benefit Degradation: {annual_degradation*100}%")

# Assume annual benefits are equal to the calculated ecosystem service value
df['Annual_Ecosystem_Benefit'] = df['Ecosystem_Service_Value']
df['Initial_Investment_K'] = df['Rehabilitation_Investment_M'] * 1000  # Convert to thousands

# Calculate NPV without degradation
df['NPV_No_Degradation'] = df.apply(
    lambda row: calculate_npv(
        row['Initial_Investment_K'], 
        row['Annual_Ecosystem_Benefit'], 
        discount_rate, 
        project_lifespan, 
        0
    ), axis=1
)

# Calculate NPV with degradation
df['NPV_With_Degradation'] = df.apply(
    lambda row: calculate_npv(
        row['Initial_Investment_K'], 
        row['Annual_Ecosystem_Benefit'], 
        discount_rate, 
        project_lifespan, 
        annual_degradation
    ), axis=1
)

# NPV Analysis Results
print("\n1. NPV Analysis Results:")
print("\nWithout Benefit Degradation:")
npv_stats_no_deg = df['NPV_No_Degradation'].describe()
print(npv_stats_no_deg.round(2))
print(f"Percentage of projects with positive NPV: {(df['NPV_No_Degradation'] > 0).mean() * 100:.1f}%")

print("\nWith Benefit Degradation:")
npv_stats_with_deg = df['NPV_With_Degradation'].describe()
print(npv_stats_with_deg.round(2))
print(f"Percentage of projects with positive NPV: {(df['NPV_With_Degradation'] > 0).mean() * 100:.1f}%")

# NPV by mine
print("\n2. NPV Analysis by Mine:")
npv_by_mine = df.groupby('Mine')[['NPV_No_Degradation', 'NPV_With_Degradation']].agg(['mean', 'std']).round(2)
print(npv_by_mine)

# Sensitivity analysis - different discount rates
discount_rates = [0.03, 0.05, 0.07, 0.10]
sensitivity_results = []

for rate in discount_rates:
    npv_values = df.apply(
        lambda row: calculate_npv(
            row['Initial_Investment_K'], 
            row['Annual_Ecosystem_Benefit'], 
            rate, 
            project_lifespan, 
            annual_degradation
        ), axis=1
    )
    sensitivity_results.append({
        'Discount_Rate': f"{rate*100}%",
        'Mean_NPV': npv_values.mean(),
        'Positive_NPV_Pct': (npv_values > 0).mean() * 100
    })

sensitivity_df = pd.DataFrame(sensitivity_results)
print("\n3. Sensitivity Analysis - Different Discount Rates:")
print(sensitivity_df.round(2))

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Plot 1: NPV distribution comparison
axes[0,0].hist(df['NPV_No_Degradation'], bins=20, alpha=0.5, label='No Degradation', color='green')
axes[0,0].hist(df['NPV_With_Degradation'], bins=20, alpha=0.5, label='With Degradation', color='red')
axes[0,0].axvline(0, color='black', linestyle='--', alpha=0.7, label='Break-even')
axes[0,0].set_xlabel('NPV ($K)')
axes[0,0].set_ylabel('Frequency')
axes[0,0].set_title('NPV Distribution Comparison')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Plot 2: NPV vs Investment amount
axes[0,1].scatter(df['Rehabilitation_Investment_M'], df['NPV_With_Degradation'], alpha=0.6)
axes[0,1].axhline(0, color='red', linestyle='--', alpha=0.7, label='Break-even')
axes[0,1].set_xlabel('Initial Investment ($M)')
axes[0,1].set_ylabel('NPV ($K)')
axes[0,1].set_title('NPV vs Initial Investment')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# Plot 3: Sensitivity analysis
axes[1,0].plot(sensitivity_df['Discount_Rate'], sensitivity_df['Mean_NPV'], 'o-', linewidth=2, markersize=8)
axes[1,0].axhline(0, color='red', linestyle='--', alpha=0.7, label='Break-even')
axes[1,0].set_xlabel('Discount Rate')
axes[1,0].set_ylabel('Mean NPV ($K)')
axes[1,0].set_title('NPV Sensitivity to Discount Rate')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# Plot 4: Percentage of positive NPV projects
axes[1,1].bar(sensitivity_df['Discount_Rate'], sensitivity_df['Positive_NPV_Pct'], alpha=0.7, color='orange')
axes[1,1].set_xlabel('Discount Rate')
axes[1,1].set_ylabel('% of Projects with Positive NPV')
axes[1,1].set_title('Positive NPV Projects by Discount Rate')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Economic interpretation
print("\n4. Economic Interpretation:")
print(f"Average NPV (with degradation): ${df['NPV_With_Degradation'].mean():.0f}K")
print(f"Best performing mine (highest mean NPV): {npv_by_mine['NPV_With_Degradation']['mean'].idxmax()}")
print(f"Most consistent performer (lowest NPV std): {npv_by_mine['NPV_With_Degradation']['std'].idxmin()}")

# Calculate payback period (simplified)
df['Payback_Period'] = df['Initial_Investment_K'] / df['Annual_Ecosystem_Benefit']
print(f"\nAverage Payback Period: {df['Payback_Period'].mean():.1f} years")
print(f"Median Payback Period: {df['Payback_Period'].median():.1f} years")

## 6. Econometric Methods

### 6.1 Panel Data Models

In [ ]:
# Panel Data Analysis using Fixed-Effects and Random-Effects Models
print("=== PANEL DATA MODELS ===")

# Install panel data tools if not available
try:
    from linearmodels import PanelOLS, RandomEffects
    panel_tools_available = True
except ImportError:
    print("Note: linearmodels not available. Installing...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'linearmodels'])
    from linearmodels import PanelOLS, RandomEffects
    panel_tools_available = True

# Prepare panel data
# Set MultiIndex for panel data (Mine, Year)
df_panel = df.set_index(['Mine', 'Year'])

print("\n1. Panel Data Structure:")
print(f"Number of mines: {df['Mine'].nunique()}")
print(f"Number of years: {df['Year'].nunique()}")
print(f"Total observations: {len(df)}")
print(f"Balanced panel: {'Yes' if len(df) == df['Mine'].nunique() * df['Year'].nunique() else 'No'}")

# Model 1: Fixed Effects Model for Investment Effectiveness
print("\n2. Fixed Effects Model: Investment Effectiveness")
print("Model: Effectiveness_Rating = α + β₁*Investment + β₂*Area + β₃*Revenue + Mine_FE + ε")

# Prepare variables for panel regression
dependent_var = df_panel['Effectiveness_Rating']
independent_vars = df_panel[['Rehabilitation_Investment_M', 'Disturbed_Area_Ha', 'Mining_Revenue_M']]

# Fixed Effects Model
fe_model = PanelOLS(dependent_var, independent_vars, entity_effects=True)
fe_results = fe_model.fit(cov_type='clustered', cluster_entity=True)
print(fe_results.summary)

# Model 2: Random Effects Model
print("\n3. Random Effects Model: Investment Effectiveness")
re_model = RandomEffects(dependent_var, independent_vars)
re_results = re_model.fit(cov_type='clustered', cluster_entity=True)
print(re_results.summary)

# Hausman Test (simplified version)
print("\n4. Model Comparison:")
print(f"Fixed Effects R-squared: {fe_results.rsquared:.4f}")
print(f"Random Effects R-squared: {re_results.rsquared:.4f}")
print(f"\nFixed Effects preferred if individual effects are correlated with regressors")
print(f"Random Effects more efficient if no correlation exists")

# Model 3: Panel model for Ecological Recovery
print("\n5. Panel Model: Ecological Recovery Index")
dependent_var2 = df_panel['Ecological_Recovery_Index']
independent_vars2 = df_panel[['Rehabilitation_Investment_M', 'Investment_Per_Area', 'Year']]

fe_model2 = PanelOLS(dependent_var2, independent_vars2, entity_effects=True, time_effects=True)
fe_results2 = fe_model2.fit(cov_type='clustered', cluster_entity=True)
print("Model with both entity and time fixed effects:")
print(fe_results2.summary)

# Analysis of mine-specific effects
print("\n6. Mine-Specific Effects Analysis:")
entity_effects = fe_results.estimated_effects
print("Entity (Mine) Fixed Effects:")
for mine, effect in entity_effects.items():
    print(f"{mine}: {effect:.3f}")

# Calculate within and between variations
mine_means = df.groupby('Mine')[['Rehabilitation_Investment_M', 'Effectiveness_Rating']].mean()
print("\n7. Between-Mine Variation:")
print(f"Investment variation between mines (std): {mine_means['Rehabilitation_Investment_M'].std():.3f}")
print(f"Effectiveness variation between mines (std): {mine_means['Effectiveness_Rating'].std():.3f}")

# Within-mine variation
within_var_inv = []
within_var_eff = []
for mine in df['Mine'].unique():
    mine_data = df[df['Mine'] == mine]
    within_var_inv.append(mine_data['Rehabilitation_Investment_M'].std())
    within_var_eff.append(mine_data['Effectiveness_Rating'].std())

print("\n8. Within-Mine Variation:")
print(f"Average within-mine investment variation (std): {np.mean(within_var_inv):.3f}")
print(f"Average within-mine effectiveness variation (std): {np.mean(within_var_eff):.3f}")

# Visualization of panel data patterns
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Plot 1: Investment trends by mine
for mine in df['Mine'].unique():
    mine_data = df[df['Mine'] == mine]
    axes[0,0].plot(mine_data['Year'], mine_data['Rehabilitation_Investment_M'], 'o-', label=mine, linewidth=2, markersize=6)
axes[0,0].set_xlabel('Year')
axes[0,0].set_ylabel('Rehabilitation Investment ($M)')
axes[0,0].set_title('Investment Trends by Mine')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Plot 2: Effectiveness trends by mine
for mine in df['Mine'].unique():
    mine_data = df[df['Mine'] == mine]
    axes[0,1].plot(mine_data['Year'], mine_data['Effectiveness_Rating'], 's-', label=mine, linewidth=2, markersize=6)
axes[0,1].set_xlabel('Year')
axes[0,1].set_ylabel('Effectiveness Rating')
axes[0,1].set_title('Effectiveness Trends by Mine')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# Plot 3: Fixed effects estimates
mines = list(entity_effects.keys())
effects = list(entity_effects.values())
colors = ['red' if e < 0 else 'green' for e in effects]
axes[1,0].bar(mines, effects, color=colors, alpha=0.7)
axes[1,0].axhline(0, color='black', linestyle='-', alpha=0.5)
axes[1,0].set_xlabel('Mine')
axes[1,0].set_ylabel('Fixed Effect')
axes[1,0].set_title('Mine-Specific Fixed Effects\n(Green=Above average, Red=Below average)')
axes[1,0].grid(True, alpha=0.3)

# Plot 4: Residuals from fixed effects model
axes[1,1].scatter(fe_results.fitted_values, fe_results.resids, alpha=0.6)
axes[1,1].axhline(0, color='red', linestyle='--')
axes[1,1].set_xlabel('Fitted Values')
axes[1,1].set_ylabel('Residuals')
axes[1,1].set_title('Fixed Effects Model: Residuals vs Fitted')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary table of all econometric models
print("\n9. Econometric Models Summary:")
econometric_summary = pd.DataFrame({
    'Model': ['Fixed Effects (Effectiveness)', 'Random Effects (Effectiveness)', 'FE with Time Effects (Ecology)'],
    'R_squared': [fe_results.rsquared, re_results.rsquared, fe_results2.rsquared],
    'N_obs': [fe_results.nobs, re_results.nobs, fe_results2.nobs],
    'Investment_Coef': [fe_results.params['Rehabilitation_Investment_M'], 
                        re_results.params['Rehabilitation_Investment_M'],
                        fe_results2.params['Rehabilitation_Investment_M']],
    'Investment_PValue': [fe_results.pvalues['Rehabilitation_Investment_M'],
                          re_results.pvalues['Rehabilitation_Investment_M'],
                          fe_results2.pvalues['Rehabilitation_Investment_M']]
})
print(econometric_summary.round(4))

## 7. Summary and Conclusions

### 7.1 Key Findings Summary

In [ ]:
# Generate comprehensive summary of all analyses
print("=== COMPREHENSIVE ANALYSIS SUMMARY ===")
print("\n" + "="*70)
print("MINE REHABILITATION FINANCIAL INVESTMENT ANALYSIS")
print("OBJECTIVE 3: Financial Investment Analysis")
print("="*70)

print("\n1. DESCRIPTIVE STATISTICS SUMMARY:")
print(f"   • Total rehabilitation investment across all projects: ${df['Rehabilitation_Investment_M'].sum():.1f}M")
print(f"   • Average investment per project: ${df['Rehabilitation_Investment_M'].mean():.2f}M")
print(f"   • Average investment per hectare: ${df['Investment_Per_Area'].mean():.1f}K")
print(f"   • Average investment as % of revenue: {df['Investment_Revenue_Ratio'].mean():.1f}%")
print(f"   • Average investment as % of profit: {df['Investment_Profit_Ratio'].mean():.1f}%")

print("\n2. TREND ANALYSIS SUMMARY:")
yearly_growth = yearly_data['Investment_Growth_Rate'].dropna().mean()
print(f"   • Annual trend in total investments: {'Increasing' if slope > 0 else 'Decreasing'} (${slope:.2f}M/year)")
print(f"   • Average year-over-year growth rate: {yearly_growth:.1f}%")
print(f"   • Investment time series is: {'Stationary' if adf_result[1] < 0.05 else 'Non-stationary'}")

print("\n3. CORRELATION ANALYSIS SUMMARY:")
key_correlations = {
    'Investment vs Effectiveness': correlation_matrix.loc['Rehabilitation_Investment_M', 'Effectiveness_Rating'],
    'Investment vs Vegetation Recovery': correlation_matrix.loc['Rehabilitation_Investment_M', 'Vegetation_Recovery'],
    'Investment vs Soil Quality': correlation_matrix.loc['Rehabilitation_Investment_M', 'Soil_Quality_Index'],
    'Investment vs Ecosystem Value': correlation_matrix.loc['Rehabilitation_Investment_M', 'Ecosystem_Service_Value']
}
for relationship, corr in key_correlations.items():
    strength = 'Strong' if abs(corr) > 0.7 else 'Moderate' if abs(corr) > 0.4 else 'Weak'
    direction = 'positive' if corr > 0 else 'negative'
    print(f"   • {relationship}: {strength} {direction} correlation ({corr:.3f})")

print("\n4. REGRESSION ANALYSIS SUMMARY:")
print(f"   • Effectiveness rating model R²: {model1.rsquared:.3f}")
print(f"   • Ecological recovery model R²: {model2.rsquared:.3f}")
print(f"   • Ecosystem service value model R²: {model3.rsquared:.3f}")
print(f"   • Investment significantly predicts effectiveness: {'Yes' if model1.pvalues['Rehabilitation_Investment_M'] < 0.05 else 'No'}")

print("\n5. COST-BENEFIT ANALYSIS SUMMARY:")
print(f"   • Average benefit-cost ratio: {df['Benefit_Cost_Ratio'].mean():.2f}")
print(f"   • Projects with B/C > 1: {(df['Benefit_Cost_Ratio'] > 1).mean() * 100:.1f}%")
print(f"   • Average payback period: {df['Payback_Period'].mean():.1f} years")

print("\n6. NET PRESENT VALUE ANALYSIS SUMMARY:")
print(f"   • Average NPV (with degradation): ${df['NPV_With_Degradation'].mean():.0f}K")
print(f"   • Projects with positive NPV: {(df['NPV_With_Degradation'] > 0).mean() * 100:.1f}%")
print(f"   • NPV sensitivity to discount rate: High (varies from ${sensitivity_df.iloc[0]["Mean_NPV"]:.0f}K to ${sensitivity_df.iloc[-1]["Mean_NPV"]:.0f}K)")

print("\n7. PANEL DATA ANALYSIS SUMMARY:")
print(f"   • Fixed effects model R²: {fe_results.rsquared:.3f}")
print(f"   • Random effects model R²: {re_results.rsquared:.3f}")
print(f"   • Significant mine-specific heterogeneity: Yes")
print(f"   • Investment effect in panel model: {fe_results.params['Rehabilitation_Investment_M']:.3f}")
print(f"   • Statistical significance: {'Yes' if fe_results.pvalues['Rehabilitation_Investment_M'] < 0.05 else 'No'}")

print("\n8. POLICY RECOMMENDATIONS:")
high_bc_mines = df.groupby('Mine')['Benefit_Cost_Ratio'].mean().sort_values(ascending=False)
best_mine = high_bc_mines.index[0]
worst_mine = high_bc_mines.index[-1]

print(f"   • Focus additional investments on: {best_mine} (highest B/C ratio: {high_bc_mines.iloc[0]:.2f})")
print(f"   • Review investment strategy for: {worst_mine} (lowest B/C ratio: {high_bc_mines.iloc[-1]:.2f})")
print(f"   • Optimal investment level appears to be around: ${df['Rehabilitation_Investment_M'].median():.1f}M per project")
print(f"   • Investment effectiveness varies significantly across mines - customize approaches")

if df['Benefit_Cost_Ratio'].mean() > 1:
    print("   • Overall assessment: Rehabilitation investments are economically justified")
else:
    print("   • Overall assessment: Review investment strategies to improve cost-effectiveness")

print("\n9. DATA LIMITATIONS AND FUTURE RESEARCH:")
print("   • This analysis uses simulated data for demonstration purposes")
print("   • Real-world application would require:")
print("     - Actual financial investment data")
print("     - Measured ecological recovery indicators")
print("     - Long-term monitoring data")
print("     - External factors (regulations, market conditions)")
print("   • Recommended future research:")
print("     - Include risk analysis and uncertainty quantification")
print("     - Incorporate environmental externalities")
print("     - Analyze optimal timing of investments")
print("     - Consider technological innovations in rehabilitation")

print("\n" + "="*70)
print("END OF ANALYSIS")
print("="*70)

# Export summary statistics to a DataFrame for easy reference
summary_stats = pd.DataFrame({
    'Metric': [
        'Total Investment ($M)',
        'Average Investment per Project ($M)',
        'Average Investment per Hectare ($K)',
        'Average B/C Ratio',
        'Projects with Positive NPV (%)',
        'Average Payback Period (years)',
        'Investment-Effectiveness Correlation',
        'Panel Model R-squared'
    ],
    'Value': [
        f"{df['Rehabilitation_Investment_M'].sum():.1f}",
        f"{df['Rehabilitation_Investment_M'].mean():.2f}",
        f"{df['Investment_Per_Area'].mean():.1f}",
        f"{df['Benefit_Cost_Ratio'].mean():.2f}",
        f"{(df['NPV_With_Degradation'] > 0).mean() * 100:.1f}",
        f"{df['Payback_Period'].mean():.1f}",
        f"{correlation_matrix.loc['Rehabilitation_Investment_M', 'Effectiveness_Rating']:.3f}",
        f"{fe_results.rsquared:.3f}"
    ]
})

print("\n\nSUMMARY STATISTICS TABLE:")
print(summary_stats.to_string(index=False))